In [1]:
import numpy as np
import ot
from sklearn.preprocessing import StandardScaler

In [17]:
def RNAsc_builder(opt, data):
    assert opt['problem_name'] == 'RNAsc'
    _dict = data  
    datas = _dict['pcs']
    scaler = StandardScaler()
    scaler.fit(datas)
    datas = scaler.transform(datas)
    datas = datas[:, 0:opt['RNA_dim']]
    datas = datas.astype('float32')
    timestamps = _dict['sample_labels']
    tokens = np.arange(0, opt['num_marg'])
    datasets = [datas[np.where(token == timestamps)] for token in tokens]
   
    return datasets

In [23]:
import torch
from torch import nn


class RBF(nn.Module):

    def __init__(self, n_kernels=5, mul_factor=2.0, bandwidth=None):
        super().__init__()
        self.bandwidth_multipliers = mul_factor ** (torch.arange(n_kernels) - n_kernels // 2)
        self.bandwidth = bandwidth

    def get_bandwidth(self, L2_distances):
        if self.bandwidth is None:
            n_samples = L2_distances.shape[0]
            return L2_distances.data.sum() / (n_samples ** 2 - n_samples)

        return self.bandwidth

    def forward(self, X):
        L2_distances = torch.cdist(X, X) ** 2
        return torch.exp(-L2_distances[None, ...] / (self.get_bandwidth(L2_distances) * self.bandwidth_multipliers)[:, None, None]).sum(dim=0)


class MMDLoss(nn.Module):

    def __init__(self, kernel=RBF()):
        super().__init__()
        self.kernel = kernel

    def forward(self, X, Y):
        K = self.kernel(torch.vstack([X, Y]))

        X_size = X.shape[0]
        XX = K[:X_size, :X_size].mean()
        XY = K[:X_size, X_size:].mean()
        YY = K[X_size:, X_size:].mean()
        return XX - 2 * XY + YY

In [18]:
def sliced_wasserstein_with_n_proj(x_np, y_np, n_proj=50):
    return ot.sliced.sliced_wasserstein_distance(x_np, y_np, n_projections=n_proj)

In [19]:
def sliced_wasserstein_sampling(datasets, n_samples=500, n_repeats=10000, n_proj=50):
    """
    Pour chaque dataset dans la liste, tire aléatoirement 2x500 points (sans remplacement),
    calcule la sliced wasserstein distance entre les deux sous-ensembles, répète n_repeats fois,
    et retourne la moyenne et l'écart-type pour chaque dataset.
    """
    results = []
    for idx, data in enumerate(datasets):
        distances = []
        n_points = data.shape[0]
        for rep in range(n_repeats):
            rng = np.random.default_rng(seed=rep)
            indices = rng.permutation(n_points)
            if n_points < 1000:
                raise ValueError(f"Le dataset {idx} n'a pas assez de points pour tirer 2x{n_samples} échantillons uniques.")
            idx1 = indices[:n_samples]
            idx2 = indices[n_samples:2*n_samples]
            x1 = data[idx1]
            x2 = data[idx2]
            dist = sliced_wasserstein_with_n_proj(x1, x2, n_proj=n_proj)
            distances.append(dist)
        mean_dist = np.mean(distances)
        std_dist = np.std(distances)
        results.append({'dataset_idx': idx, 'mean_sliced': mean_dist, 'std_sliced': std_dist})
    return results

In [25]:
def mmd_sampling(datasets, n_samples=500, n_repeats=10000, kernel=None):
    """
    Pour chaque dataset dans la liste, tire aléatoirement 2x n_samples points (sans remplacement),
    calcule la MMD entre les deux sous-ensembles, répète n_repeats fois,
    et retourne la moyenne et l'écart-type pour chaque dataset.
    """
    if kernel is None:
        kernel = MMDLoss()  # Utilise le kernel par défaut (RBF)
    results = []
    for idx, data in enumerate(datasets):
        distances = []
        n_points = data.shape[0]
        if n_points < 2 * n_samples:
            raise ValueError(f"Le dataset {idx} n'a pas assez de points pour tirer 2x{n_samples} échantillons uniques.")
        for rep in range(n_repeats):
            rng = np.random.default_rng(seed=rep)
            indices = rng.permutation(n_points)
            idx1 = indices[:n_samples]
            idx2 = indices[n_samples:2*n_samples]
            x1 = torch.from_numpy(data[idx1])
            x2 = torch.from_numpy(data[idx2])
            dist = kernel(x1, x2).item()
            distances.append(dist)
        mean_dist = np.mean(distances)
        std_dist = np.std(distances)
        results.append({'dataset_idx': idx, 'mean_mmd': mean_dist, 'std_mmd': std_dist})
    return results

In [30]:
def max_sliced_wasserstein_with_n_proj(x_np, y_np, n_proj=50):
    # Compute the maximum sliced Wasserstein distance over n_proj random projections
    return ot.sliced.max_sliced_wasserstein_distance(x_np, y_np, n_projections=n_proj)

def max_sliced_wasserstein_sampling(datasets, n_samples=500, n_repeats=10000, n_proj=50):
    """
    Pour chaque dataset dans la liste, tire aléatoirement 2x500 points (sans remplacement),
    calcule la max sliced wasserstein distance entre les deux sous-ensembles, répète n_repeats fois,
    et retourne la moyenne et l'écart-type pour chaque dataset.
    """
    results = []
    for idx, data in enumerate(datasets):
        distances = []
        n_points = data.shape[0]
        for rep in range(n_repeats):
            rng = np.random.default_rng(seed=rep)
            indices = rng.permutation(n_points)
            if n_points < 1000:
                raise ValueError(f"Le dataset {idx} n'a pas assez de points pour tirer 2x{n_samples} échantillons uniques.")
            idx1 = indices[:n_samples]
            idx2 = indices[n_samples:2*n_samples]
            x1 = data[idx1]
            x2 = data[idx2]
            dist = max_sliced_wasserstein_with_n_proj(x1, x2, n_proj=n_proj)
            distances.append(dist)
        mean_dist = np.mean(distances)
        std_dist = np.std(distances)
        results.append({'dataset_idx': idx, 'mean_max_sliced': mean_dist, 'std_max_sliced': std_dist})
    return results

## Result for 100 dim

In [20]:
opt = {
    'problem_name': 'RNAsc',
    'RNA_dim': 100,      
    'num_marg': 5         
}

### SWD

#### 500 samples

In [21]:
datasets = RNAsc_builder(opt, data)
results = sliced_wasserstein_sampling(datasets, n_samples = 500, n_repeats= 1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_sliced': 0.0834436873330194, 'std_sliced': 0.004534384012476335}, {'dataset_idx': 1, 'mean_sliced': 0.10425739165557092, 'std_sliced': 0.005242565776722317}, {'dataset_idx': 2, 'mean_sliced': 0.11594053026419546, 'std_sliced': 0.005680501510770535}, {'dataset_idx': 3, 'mean_sliced': 0.12099236538087661, 'std_sliced': 0.006135370837170378}, {'dataset_idx': 4, 'mean_sliced': 0.13353555249551577, 'std_sliced': 0.006669101798488719}]


#### 1000 samples

In [22]:
datasets = RNAsc_builder(opt, data)
results = sliced_wasserstein_sampling(datasets, n_samples = 1000, n_repeats= 1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_sliced': 0.06044912349378335, 'std_sliced': 0.0033296201891726772}, {'dataset_idx': 1, 'mean_sliced': 0.07523775533136205, 'std_sliced': 0.00351759868787148}, {'dataset_idx': 2, 'mean_sliced': 0.08421555763613796, 'std_sliced': 0.0038681209449125337}, {'dataset_idx': 3, 'mean_sliced': 0.08770115465059114, 'std_sliced': 0.004230280960825873}, {'dataset_idx': 4, 'mean_sliced': 0.09650604455518894, 'std_sliced': 0.0043710383618197}]


### MMD


#### 500 samples

In [28]:
results = mmd_sampling(datasets, n_samples=500, n_repeats=100)
print(results)

[{'dataset_idx': 0, 'mean_mmd': 0.012063286304473876, 'std_mmd': 0.001092381859913278}, {'dataset_idx': 1, 'mean_mmd': 0.012286943197250367, 'std_mmd': 0.0009136084792118105}, {'dataset_idx': 2, 'mean_mmd': 0.012181066274642945, 'std_mmd': 0.0007152476854059872}, {'dataset_idx': 3, 'mean_mmd': 0.012302075624465942, 'std_mmd': 0.000722412820543576}, {'dataset_idx': 4, 'mean_mmd': 0.0121684730052948, 'std_mmd': 0.0007310168496801934}]


#### 1000 samples

In [29]:
results = mmd_sampling(datasets, n_samples=1000, n_repeats=100)
print(results)

[{'dataset_idx': 0, 'mean_mmd': 0.006043922901153564, 'std_mmd': 0.0005628068236331856}, {'dataset_idx': 1, 'mean_mmd': 0.00616602897644043, 'std_mmd': 0.00039030410906089693}, {'dataset_idx': 2, 'mean_mmd': 0.006090550422668457, 'std_mmd': 0.00039497387764395335}, {'dataset_idx': 3, 'mean_mmd': 0.006069717407226563, 'std_mmd': 0.000369316936820107}, {'dataset_idx': 4, 'mean_mmd': 0.006120337247848511, 'std_mmd': 0.00036903623247168003}]


### Max SWD

#### 500 samples

In [ ]:
results = max_sliced_wasserstein_sampling(datasets, n_samples=500, n_repeats=1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_max_sliced': 0.14199171906207397, 'std_max_sliced': 0.01874690448867213}, {'dataset_idx': 1, 'mean_max_sliced': 0.1759949855382898, 'std_max_sliced': 0.021597665860281397}, {'dataset_idx': 2, 'mean_max_sliced': 0.19633453950568072, 'std_max_sliced': 0.023879866186924125}, {'dataset_idx': 3, 'mean_max_sliced': 0.20694248330154222, 'std_max_sliced': 0.030107965848898365}, {'dataset_idx': 4, 'mean_max_sliced': 0.2254343812967639, 'std_max_sliced': 0.02727976697415563}]


#### 1000 samples

In [32]:
results = max_sliced_wasserstein_sampling(datasets, n_samples=1000, n_repeats=1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_max_sliced': 0.10135066081989881, 'std_max_sliced': 0.013179227062546072}, {'dataset_idx': 1, 'mean_max_sliced': 0.1250938407776347, 'std_max_sliced': 0.015704811442606742}, {'dataset_idx': 2, 'mean_max_sliced': 0.140532449707046, 'std_max_sliced': 0.017254952707299556}, {'dataset_idx': 3, 'mean_max_sliced': 0.14895741713814198, 'std_max_sliced': 0.021421113541554192}, {'dataset_idx': 4, 'mean_max_sliced': 0.16114587711273493, 'std_max_sliced': 0.019013208557969945}]


In [ ]:
opt = {
    'problem_name': 'RNAsc',
    'RNA_dim': 5,      
    'num_marg': 5         
}

## Result for 5 dim

In [33]:
opt = {
    'problem_name': 'RNAsc',
    'RNA_dim': 5,      
    'num_marg': 5         
}

### SWD

#### 500 samples

In [34]:
datasets = RNAsc_builder(opt, data)
results = sliced_wasserstein_sampling(datasets, n_samples = 500, n_repeats= 1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_sliced': 0.06036729106828641, 'std_sliced': 0.008825033477280995}, {'dataset_idx': 1, 'mean_sliced': 0.08103643112547007, 'std_sliced': 0.012403494629809044}, {'dataset_idx': 2, 'mean_sliced': 0.10213192882349315, 'std_sliced': 0.019296585400829413}, {'dataset_idx': 3, 'mean_sliced': 0.09958631032347313, 'std_sliced': 0.016277241180549903}, {'dataset_idx': 4, 'mean_sliced': 0.09997338568706828, 'std_sliced': 0.016322375659918294}]


#### 1000 samples

In [35]:
datasets = RNAsc_builder(opt, data)
results = sliced_wasserstein_sampling(datasets, n_samples = 1000, n_repeats= 1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_sliced': 0.043965641451664275, 'std_sliced': 0.006573473698418558}, {'dataset_idx': 1, 'mean_sliced': 0.05792900037993985, 'std_sliced': 0.008794138289854925}, {'dataset_idx': 2, 'mean_sliced': 0.07414043515062076, 'std_sliced': 0.012915997407647317}, {'dataset_idx': 3, 'mean_sliced': 0.07086056185433334, 'std_sliced': 0.011664904264951663}, {'dataset_idx': 4, 'mean_sliced': 0.07168735471538312, 'std_sliced': 0.0121320999905426}]


### MMD


#### 500 samples

In [36]:
results = mmd_sampling(datasets, n_samples=500, n_repeats=100)
print(results)

[{'dataset_idx': 0, 'mean_mmd': 0.00983980655670166, 'std_mmd': 0.0030518974348665287}, {'dataset_idx': 1, 'mean_mmd': 0.010644364356994628, 'std_mmd': 0.003949438773491405}, {'dataset_idx': 2, 'mean_mmd': 0.009940519332885742, 'std_mmd': 0.00420919990021991}, {'dataset_idx': 3, 'mean_mmd': 0.010893006324768067, 'std_mmd': 0.004315606537253189}, {'dataset_idx': 4, 'mean_mmd': 0.010689592361450196, 'std_mmd': 0.004371593177397985}]


#### 1000 samples

In [37]:
results = mmd_sampling(datasets, n_samples=1000, n_repeats=100)
print(results)

[{'dataset_idx': 0, 'mean_mmd': 0.005273933410644531, 'std_mmd': 0.0020128282235805047}, {'dataset_idx': 1, 'mean_mmd': 0.005357844829559326, 'std_mmd': 0.0019356804264184675}, {'dataset_idx': 2, 'mean_mmd': 0.005121159553527832, 'std_mmd': 0.002086394610033962}, {'dataset_idx': 3, 'mean_mmd': 0.005251455307006836, 'std_mmd': 0.0025831176242943218}, {'dataset_idx': 4, 'mean_mmd': 0.005110759735107422, 'std_mmd': 0.001926202740511994}]


### Max SWD

#### 500 samples

In [38]:
results = max_sliced_wasserstein_sampling(datasets, n_samples=500, n_repeats=1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_max_sliced': 0.09992974735807433, 'std_max_sliced': 0.019076505849142833}, {'dataset_idx': 1, 'mean_max_sliced': 0.1331096435559385, 'std_max_sliced': 0.027156244286801454}, {'dataset_idx': 2, 'mean_max_sliced': 0.1720672174614116, 'std_max_sliced': 0.04323412862582893}, {'dataset_idx': 3, 'mean_max_sliced': 0.16586571504532738, 'std_max_sliced': 0.03613849532101426}, {'dataset_idx': 4, 'mean_max_sliced': 0.1669847155870052, 'std_max_sliced': 0.03657237496594846}]


#### 1000 samples

In [39]:
results = max_sliced_wasserstein_sampling(datasets, n_samples=1000, n_repeats=1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_max_sliced': 0.0727381066838198, 'std_max_sliced': 0.01404116551266604}, {'dataset_idx': 1, 'mean_max_sliced': 0.09487748385134986, 'std_max_sliced': 0.019489849873917733}, {'dataset_idx': 2, 'mean_max_sliced': 0.12489503641774044, 'std_max_sliced': 0.031042244473814814}, {'dataset_idx': 3, 'mean_max_sliced': 0.11677235399235367, 'std_max_sliced': 0.025222177442551584}, {'dataset_idx': 4, 'mean_max_sliced': 0.11929639529118864, 'std_max_sliced': 0.02818739211312376}]


## Result for 50 dim

In [40]:
opt = {
    'problem_name': 'RNAsc',
    'RNA_dim': 50 ,      
    'num_marg': 5         
}

### SWD

#### 500 samples

In [41]:
datasets = RNAsc_builder(opt, data)
results = sliced_wasserstein_sampling(datasets, n_samples = 500, n_repeats= 1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_sliced': 0.08463989455970528, 'std_sliced': 0.0067517852261526085}, {'dataset_idx': 1, 'mean_sliced': 0.10604077648109123, 'std_sliced': 0.006619798070685683}, {'dataset_idx': 2, 'mean_sliced': 0.11962157259508284, 'std_sliced': 0.007141769914854723}, {'dataset_idx': 3, 'mean_sliced': 0.12491103940270157, 'std_sliced': 0.008263070063621559}, {'dataset_idx': 4, 'mean_sliced': 0.12855743601389008, 'std_sliced': 0.008084657685507288}]


#### 1000 samples

In [42]:
datasets = RNAsc_builder(opt, data)
results = sliced_wasserstein_sampling(datasets, n_samples = 1000, n_repeats= 1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_sliced': 0.0615335050585972, 'std_sliced': 0.00467003466503916}, {'dataset_idx': 1, 'mean_sliced': 0.07687306102202572, 'std_sliced': 0.0046973849523364725}, {'dataset_idx': 2, 'mean_sliced': 0.08713156284080772, 'std_sliced': 0.0047908177083182905}, {'dataset_idx': 3, 'mean_sliced': 0.09016620078665158, 'std_sliced': 0.005525640024068094}, {'dataset_idx': 4, 'mean_sliced': 0.09294114866119807, 'std_sliced': 0.005553805203121051}]


### MMD


#### 500 samples

In [43]:
results = mmd_sampling(datasets, n_samples=500, n_repeats=100)
print(results)

[{'dataset_idx': 0, 'mean_mmd': 0.011793694496154784, 'std_mmd': 0.001929222634609894}, {'dataset_idx': 1, 'mean_mmd': 0.012186609506607056, 'std_mmd': 0.0014611525574905037}, {'dataset_idx': 2, 'mean_mmd': 0.0118747878074646, 'std_mmd': 0.0011134121998470474}, {'dataset_idx': 3, 'mean_mmd': 0.012270470857620239, 'std_mmd': 0.0011280720174393177}, {'dataset_idx': 4, 'mean_mmd': 0.01207376480102539, 'std_mmd': 0.0011802445309515366}]


#### 1000 samples

In [44]:
results = mmd_sampling(datasets, n_samples=1000, n_repeats=100)
print(results)

[{'dataset_idx': 0, 'mean_mmd': 0.005927371978759766, 'std_mmd': 0.0009726437496650651}, {'dataset_idx': 1, 'mean_mmd': 0.006059935092926025, 'std_mmd': 0.0006611423684968111}, {'dataset_idx': 2, 'mean_mmd': 0.00599368691444397, 'std_mmd': 0.0005293335683128376}, {'dataset_idx': 3, 'mean_mmd': 0.006059401035308838, 'std_mmd': 0.0006298335223411526}, {'dataset_idx': 4, 'mean_mmd': 0.00606365442276001, 'std_mmd': 0.0006138923606480788}]


### Max SWD

#### 500 samples

In [45]:
results = max_sliced_wasserstein_sampling(datasets, n_samples=500, n_repeats=1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_max_sliced': 0.14859525994973227, 'std_max_sliced': 0.02472005490269436}, {'dataset_idx': 1, 'mean_max_sliced': 0.18248912606317974, 'std_max_sliced': 0.027658570645393683}, {'dataset_idx': 2, 'mean_max_sliced': 0.20645784264321554, 'std_max_sliced': 0.03142127468442036}, {'dataset_idx': 3, 'mean_max_sliced': 0.2178757910294581, 'std_max_sliced': 0.0363612036761672}, {'dataset_idx': 4, 'mean_max_sliced': 0.22224034985504362, 'std_max_sliced': 0.034713041169249836}]


#### 1000 samples

In [46]:
results = max_sliced_wasserstein_sampling(datasets, n_samples=1000, n_repeats=1000, n_proj=50)
print(results)

[{'dataset_idx': 0, 'mean_max_sliced': 0.10643743619773854, 'std_max_sliced': 0.017829738381880504}, {'dataset_idx': 1, 'mean_max_sliced': 0.13021889433209025, 'std_max_sliced': 0.02003598476678519}, {'dataset_idx': 2, 'mean_max_sliced': 0.14759293396911438, 'std_max_sliced': 0.021497275286762985}, {'dataset_idx': 3, 'mean_max_sliced': 0.15853440484337375, 'std_max_sliced': 0.029032548142294612}, {'dataset_idx': 4, 'mean_max_sliced': 0.15849372750121146, 'std_max_sliced': 0.02444826378002088}]


In [48]:
import pandas as pd

def compute_all_metrics(data, dims, n_samples_list, n_repeats_swd=1000, n_repeats_mmd=100, n_proj=50, num_marg=5):
    """
    Calcule SWD, Max SWD et MMD pour différentes dimensions et tailles d'échantillons.
    Retourne un DataFrame à double entrée (dim, n_samples) avec les résultats pour chaque dataset.
    """

    results_table = []

    for dim in dims:
        opt = {
            'problem_name': 'RNAsc',
            'RNA_dim': dim,
            'num_marg': num_marg
        }
        datasets = RNAsc_builder(opt, data)
        for n_samples in n_samples_list:
            # SWD
            swd = sliced_wasserstein_sampling(datasets, n_samples=n_samples, n_repeats=n_repeats_swd, n_proj=n_proj)
            # Max SWD
            max_swd = max_sliced_wasserstein_sampling(datasets, n_samples=n_samples, n_repeats=n_repeats_swd, n_proj=n_proj)
            # MMD
            mmd = mmd_sampling(datasets, n_samples=n_samples, n_repeats=n_repeats_mmd)

            for idx in range(len(datasets)):
                results_table.append({
                    'dim': dim,
                    'n_samples': n_samples,
                    'dataset_idx': idx,
                    'mean_swd': swd[idx]['mean_sliced'],
                    'std_swd': swd[idx]['std_sliced'],
                    'mean_max_swd': max_swd[idx]['mean_max_sliced'],
                    'std_max_swd': max_swd[idx]['std_max_sliced'],
                    'mean_mmd': mmd[idx]['mean_mmd'],
                    'std_mmd': mmd[idx]['std_mmd'],
                })

    df_results = pd.DataFrame(results_table)
    return df_results

In [49]:
dims = [5, 50, 100]
n_samples_list = [500, 1000]
df_results = compute_all_metrics(data, dims, n_samples_list)

In [51]:
df_results_pivot = df_results.set_index(['dataset_idx', 'dim', 'n_samples']).sort_index()
df_results_pivot

mean_swd   std_swd  mean_max_swd  std_max_swd  \
dataset_idx dim n_samples                                                  
0           5   500        0.060373  0.008833      0.100422     0.018670   
                1000       0.043972  0.006437      0.072465     0.014219   
            50  500        0.084762  0.006253      0.148088     0.023807   
                1000       0.061474  0.004800      0.106674     0.018087   
            100 500        0.083281  0.004458      0.142880     0.018547   
                1000       0.060442  0.003171      0.101422     0.013273   
1           5   500        0.080917  0.012356      0.133198     0.026844   
                1000       0.058059  0.008879      0.094799     0.019316   
            50  500        0.105894  0.006644      0.183203     0.027961   
                1000       0.076796  0.004808      0.130367     0.019161   
            100 500        0.103921  0.005199      0.176132     0.021801   
                1000       0.075215  0.003571      0.125422     0.015405   
2           5   500        0.102302  0.019311      0.172902     0.043634   
                1000       0.074118  0.012723      0.124231     0.030017   
            50  500        0.119220  0.007263      0.204596     0.030511   
                1000       0.086954  0.004914      0.148291     0.021347   
            100 500        0.116178  0.005559      0.197003     0.024999   
                1000       0.084384  0.003949      0.141368     0.017764   
3           5   500        0.099545  0.015990      0.165687     0.035766   
                1000       0.070624  0.011391      0.116649     0.025284   
            50  500        0.124779  0.008117      0.219757     0.034784   
                1000       0.090186  0.005492      0.156819     0.027813   
            100 500        0.120737  0.006529      0.207717     0.033428   
                1000       0.087632  0.004328      0.148875     0.022060   
4           5   500        0.099830  0.016443      0.166504     0.036675   
                1000       0.071778  0.012251      0.119445     0.027963   
            50  500        0.128700  0.007854      0.219032     0.032859   
                1000       0.092985  0.005621      0.157587     0.023991   
            100 500        0.133058  0.006131      0.227025     0.029006   
                1000       0.096331  0.004445      0.160608     0.019431   

                           mean_mmd   std_mmd  
dataset_idx dim n_samples                      
0           5   500        0.009840  0.003052  
                1000       0.005274  0.002013  
            50  500        0.011794  0.001929  
                1000       0.005927  0.000973  
            100 500        0.012063  0.001092  
                1000       0.006044  0.000563  
1           5   500        0.010644  0.003949  
                1000       0.005358  0.001936  
            50  500        0.012187  0.001461  
                1000       0.006060  0.000661  
            100 500        0.012287  0.000914  
                1000       0.006166  0.000390  
2           5   500        0.009941  0.004209  
                1000       0.005121  0.002086  
            50  500        0.011875  0.001113  
                1000       0.005994  0.000529  
            100 500        0.012181  0.000715  
                1000       0.006091  0.000395  
3           5   500        0.010893  0.004316  
                1000       0.005251  0.002583  
            50  500        0.012270  0.001128  
                1000       0.006059  0.000630  
            100 500        0.012302  0.000722  
                1000       0.006070  0.000369  
4           5   500        0.010690  0.004372  
                1000       0.005111  0.001926  
            50  500        0.012074  0.001180  
                1000       0.006064  0.000614  
            100 500        0.012168  0.000731  
                1000       0.006120  0.000369

In [58]:
with open("metrics_results_pivot.tex", "w") as f:
    f.write(df_results_pivot.to_latex(multicolumn=True, multirow=True, float_format="%.3f"))

In [53]:
with pd.ExcelWriter("metrics_results.xlsx") as writer:
    # Export du tableau formaté avec titres
    df_pretty.to_excel(writer, sheet_name="Résumé formaté", index=False)
    df_results.to_excel(writer, sheet_name="Données brutes", index=False)
    df_results_pivot.to_excel(writer, sheet_name="Pivot multi-index")

ModuleNotFoundError: No module named 'openpyxl'